Geographic replication of terrain characteristics

Continue the adaptive EDA with a collection question: which measured terrain
characteristics have counterparts at several Earth locations? This inventory
does not train a model or turn scalar descriptors into geological labels.
Read the [region research](../diagnostics/region-diversity-20261029/REGION_RESEARCH.md)
and [prospective collection plan](../../EDA_PLAN.md) alongside the results.

In [ ]:
import hashlib
import itertools
import json
import sys
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.figure import Figure

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'outline.md').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
from inspection import check_input, file_hash

PREVIOUS = ROOT / 'notebooks/diagnostics/scale-eda-20261029T012946475712Z'
RETRIEVAL = ROOT / 'notebooks/diagnostics/retrieval-eda-20261029T024445352232Z'
NOTEBOOK = ROOT / 'notebooks/exploration/18_region_diversity_eda.ipynb'
OUT = ROOT / 'notebooks/diagnostics' / ('diversity-eda-' + datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ'))
OUT.mkdir(parents=True, exist_ok=False)
expected: dict[str, str] = {}
evidence: dict[str, str] = {}
for previous in [PREVIOUS, RETRIEVAL]:
    manifest = json.loads((previous / 'provenance.json').read_text())
    assert manifest['status'] == 'complete'
    expected.update({str((previous / name).resolve()): digest for name, digest in manifest['outputs'].items()})
    evidence[str(previous / 'provenance.json')] = file_hash(previous / 'provenance.json')
metrics = pd.read_csv(check_input(PREVIOUS / 'center_metrics.csv', expected, evidence))
site_candidates = pd.read_csv(check_input(RETRIEVAL / 'per_site_candidates.csv.gz', expected, evidence))
groups = json.loads(check_input(RETRIEVAL / 'spatial_groups.json', expected, evidence).read_text())
grids = json.loads(check_input(PREVIOUS / 'site_grids.json', expected, evidence).read_text())
FEATURES = ['slope_median_deg', 'slope_p90_deg', 'central_relief_m',
            'plane_residual_rms_m', 'plane_residual_fraction', 'local_sd_median_m']
KEYS = ['body', 'region', 'population', 'size_m']
SIZES = [300, 600, 960]
assert len(metrics) == 28284 and not metrics.duplicated(['center_id', 'size_m']).any()
assert np.isfinite(metrics[FEATURES].to_numpy()).all()
assert (metrics[FEATURES] >= 0).all().all()
assert (metrics['slope_median_deg'] <= metrics['slope_p90_deg']).all()
assert (metrics['slope_p90_deg'] <= 90).all()
assert (metrics['plane_residual_fraction'] <= 1).all()
assert metrics['common'].dtype == bool
assert set(metrics['size_m']) == set(SIZES)
assert set(metrics['population']) == {'natural_screen', 'full_context', 'image_supported'}
cohorts = {'eligible_at_size': metrics.copy(), 'common_960': metrics.loc[metrics['common']].copy()}
earth_sites = sorted(metrics.loc[metrics['body'] == 'earth', 'region'].unique())
assert len(earth_sites) == 3

# Deliberately coarse, declared scenarios, not learned classes or equivalence margins.
# Internal boundaries belong to the upper bin; final bins include physical maxima.
BIN_SCHEMES = {
    'coarse': {'slope_p90_deg': [5., 15., 30.], 'central_relief_m': [25., 75., 200.],
               'plane_residual_fraction': [.25, .5]},
    'shifted': {'slope_p90_deg': [7.5, 20., 35.], 'central_relief_m': [40., 100., 250.],
                'plane_residual_fraction': [.2, .4]},
}
MIN_COUNTS = [1, 5, 20]
source = '\n'.join(cell.source for cell in nbformat.read(NOTEBOOK, as_version=4).cells)
provenance: dict[str, Any] = {
    'status': 'running', 'notebook': str(NOTEBOOK.relative_to(ROOT)),
    'notebook_source_sha256': hashlib.sha256(source.encode()).hexdigest(),
    'helpers': {'inspection.py': file_hash(ROOT / 'notebooks/inspection.py')},
    'upstream': [str(p.relative_to(ROOT)) for p in [PREVIOUS, RETRIEVAL]],
    'cohorts': {'eligible_at_size': 'eligible notebook16 centers at this size on the same 960m lattice',
                'common_960': 'complete 960m centers at all sizes, as in notebook17'},
    'bin_schemes': BIN_SCHEMES, 'bin_boundary_policy': 'left closed, right open; last bin unbounded',
    'minimum_centers_per_site_bin': MIN_COUNTS, 'sizes_m': SIZES,
    'policy': 'whole sites remain development; all scales and lunar views remain grouped',
    'scope': 'coarse occupancy and existing exact descriptor bands; no analog labels or fitted model',
    'software': {name: version(name) for name in ['numpy', 'pandas', 'matplotlib', 'nbformat']},
}
(OUT / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .2})


def save_figure(fig: Figure, name: str) -> None:
    fig.savefig(OUT / f'{name}.png', bbox_inches='tight')
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight')
    display(Image(filename=str(OUT / f'{name}.png')))
    plt.close(fig)


print('Output:', OUT)

# Site inventory and physical distributions

The eligible-at-size cohort admits smaller windows excluded by the common
960 m requirement, but only on the existing 960 m center lattice. This is not
all possible placements. Counts do not measure independent sample size.
Source families remain confounded with planetary body: Earth FABDEM/WorldCover
versus lunar shape-from-shading/LOLA. Seven lunar site names all represent the
south-pole collection, not worldwide lunar diversity. No Mars terrain is included.

In [ ]:
inventory = metrics.groupby(KEYS).agg(eligible_centers=('center_id', 'size'),
                                     common_centers=('common', 'sum')).reset_index()
inventory['common_pct_of_eligible'] = 100 * inventory['common_centers'] / inventory['eligible_centers']
inventory['source_family'] = np.where(inventory['body'] == 'earth', 'FABDEM V1-2 / WorldCover 2021', 'PGDA104 SfS / LOLA')
inventory['group'] = inventory['body'] + ':' + inventory['region']
inventory['role'] = 'exploratory_development'
inventory.to_csv(OUT / 'site_inventory.csv', index=False)
(OUT / 'spatial_groups.json').write_text(json.dumps(groups, indent=2), encoding='utf-8')
quantiles = []
for cohort, frame in cohorts.items():
    for feature in FEATURES:
        result = frame.groupby(KEYS)[feature].quantile([.05, .5, .95]).unstack().reset_index()
        result.columns = [*KEYS, 'p05', 'median', 'p95']
        result['cohort'], result['feature'] = cohort, feature
        quantiles.append(result)
quantile_table = pd.concat(quantiles, ignore_index=True)
quantile_table.to_csv(OUT / 'descriptor_quantiles.csv', index=False)
display(inventory.loc[inventory['size_m'] == 600])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {'atacama_san_pedro': '#b15d20', 'lanzarote': '#15806e', 'northern_arizona': '#477db5'}
subset = cohorts['eligible_at_size'].loc[metrics['size_m'] == 600]
for ax, feature, label in zip(axes, ['slope_p90_deg', 'central_relief_m', 'plane_residual_fraction'],
                               ['P90 slope (degrees)', 'Central relief (m)', 'Plane residual fraction'], strict=True):
    for site in earth_sites:
        values = np.sort(subset.loc[subset['region'] == site, feature].to_numpy())
        ax.plot(values, np.arange(1, len(values) + 1) / len(values), label=site.replace('_', ' '), color=colors[site])
    for population, style in [('full_context', '--'), ('image_supported', '-')]:
        values = np.sort(subset.loc[subset['population'] == population, feature].to_numpy())
        ax.plot(values, np.arange(1, len(values) + 1) / len(values), style, color='#3b354b', label='Moon ' + population)
    ax.set(xlabel=label, ylabel='Fraction of centers', ylim=(0, 1))
axes[0].legend(fontsize=8)
fig.suptitle('600 m eligible-at-size distributions; lunar curves pool centers, not equal-weight sites')
fig.tight_layout()
save_figure(fig, '600m_descriptor_distributions')

# Replication across Earth sites

Cross P90 slope, central relief and plane-residual fraction into 48 coarse cells.
Test two boundary schemes and require 1, 5 or 20 centers per site/cell. These
choices diagnose fragility; they do not label geological terrain or establish
sufficient replication. Broad-bin occupancy can coexist with poor close matches.
Every cohort, size and lunar support population stays separate. A pooled lunar
percentage weights centers; the region rows allow separate assessment.

In [ ]:
bin_rows = []
coverage_rows = []
for cohort, frame in cohorts.items():
    for scheme, boundaries in BIN_SCHEMES.items():
        coded = frame.copy()
        bin_columns = []
        for feature, breaks in boundaries.items():
            name = feature + '_bin'
            coded[name] = np.searchsorted(breaks, coded[feature].to_numpy(), side='right')
            bin_columns.append(name)
        for (body, region, population, size), group in coded.groupby(KEYS):
            counts = group.groupby(bin_columns).size()
            for bins in itertools.product(range(4), range(4), range(3)):
                bin_rows.append({'cohort': cohort, 'scheme': scheme, 'body': body, 'region': region,
                                 'population': population, 'size_m': size,
                                 **dict(zip(bin_columns, bins, strict=True)),
                                 'centers': int(counts.get(bins, 0)), 'site_centers': len(group)})
bins_table = pd.DataFrame(bin_rows)
bins_table.to_csv(OUT / 'terrain_bin_inventory.csv', index=False)
bin_columns = [f + '_bin' for f in BIN_SCHEMES['coarse']]
for key, frame in bins_table.groupby(['cohort', 'scheme', 'size_m']):
    assert isinstance(key, tuple)
    cohort, scheme, size = key
    earth = frame.loc[frame['body'] == 'earth']
    for minimum in MIN_COUNTS:
        support = earth.assign(supported=earth['centers'] >= minimum).groupby(bin_columns)['supported'].sum()
        moon = frame.loc[frame['body'] == 'moon'].copy()
        moon['earth_sites'] = [int(support.loc[tuple(row)]) for row in moon[bin_columns].to_numpy()]
        for population, population_frame in moon.groupby('population'):
            parts = [('ALL_QUERY_WEIGHTED', population_frame), *list(population_frame.groupby('region'))]
            for region, group in parts:
                total = int(group['centers'].sum())
                for sites in range(4):
                    count = int(group.loc[group['earth_sites'] == sites, 'centers'].sum())
                    coverage_rows.append({'cohort': cohort, 'scheme': scheme, 'size_m': size,
                                          'min_centers_per_site_bin': minimum, 'population': population,
                                          'region': region, 'earth_sites': sites, 'queries': total,
                                          'centers': count, 'pct': 100 * count / total})
bin_coverage = pd.DataFrame(coverage_rows)
bin_coverage.to_csv(OUT / 'lunar_bin_replication.csv', index=False)

# Reuse all already-verified per-site band counts, without changing or refitting scores.
assert not site_candidates.duplicated(['query_id', 'scenario', 'candidate_site']).any()
assert (site_candidates.groupby(['query_id', 'scenario']).size() == 3).all()
site_candidates['site_in_band'] = site_candidates['within_band_count'] > 0
exact = site_candidates.groupby(['query_id', 'region', 'population', 'size_m', 'scenario']).agg(
    earth_sites=('site_in_band', 'sum')).reset_index()
exact.to_csv(OUT / 'query_site_replication.csv.gz', index=False, compression='gzip')
exact_rows = []
for key, population_frame in exact.groupby(['size_m', 'scenario', 'population']):
    assert isinstance(key, tuple)
    size, scenario, population = key
    for region, group in [('ALL_QUERY_WEIGHTED', population_frame), *list(population_frame.groupby('region'))]:
        for sites in range(4):
            count = int((group['earth_sites'] == sites).sum())
            exact_rows.append({'size_m': size, 'scenario': scenario, 'population': population,
                               'region': region, 'earth_sites': sites, 'queries': len(group),
                               'centers': count, 'pct': 100 * count / len(group)})
exact_summary = pd.DataFrame(exact_rows)
exact_summary.to_csv(OUT / 'exact_band_replication.csv', index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, population in zip(axes, ['full_context', 'image_supported'], strict=True):
    values = exact_summary.loc[(exact_summary['population'] == population) & (exact_summary['size_m'] == 600)
                               & (exact_summary['region'] == 'ALL_QUERY_WEIGHTED')]
    base = np.zeros(3)
    scenarios = ['baseline', 'with_planarity', 'with_local_sd']
    for sites, color in zip(range(4), ['#b54c46', '#e2ad4e', '#569b94', '#405f8e'], strict=True):
        heights = values.loc[values['earth_sites'] == sites].set_index('scenario').loc[scenarios, 'pct'].to_numpy()
        ax.bar(scenarios, heights, bottom=base, label=f'{sites} Earth sites', color=color)
        base += heights
    ax.set(title=population.replace('_', ' '), ylabel='Lunar queries (%)', ylim=(0, 100))
    ax.tick_params(axis='x', labelrotation=15)
axes[1].legend(fontsize=8, loc='upper right')
fig.suptitle('600 m common-center cohort: number of Earth sites with a candidate inside every band')
fig.tight_layout()
save_figure(fig, '600m_exact_site_replication')

# Where the descriptor gaps occur

Maps use existing projected center coordinates with equal aspect. Colors show
P90 slope, not a terrain class or prediction. Hollow circles on Moon maps mark
strict-view eligible centers, so support-driven selection remains visible.
Coordinates are site-local kilometres; maps are not a shared geographic CRS.

In [ ]:
for body, population in [('earth', 'natural_screen'), ('moon', 'full_context')]:
    regions = sorted(metrics.loc[metrics['body'] == body, 'region'].unique())
    fig, axes = plt.subplots(2 if body == 'moon' else 1, 4 if body == 'moon' else 3,
                             figsize=(16, 8 if body == 'moon' else 5), squeeze=False)
    for ax, region in zip(axes.flat, regions, strict=False):
        group = subset.loc[(subset['region'] == region) & (subset['population'] == population)]
        grid = grids[region]
        origin_x, origin_y = grid['transform'][2], grid['transform'][5]
        x = (group['center_x_m'] - origin_x) / 1000
        y = (group['center_y_m'] - origin_y) / 1000
        ax.scatter(x, y, c=group['slope_p90_deg'], s=8, vmin=0, vmax=40, cmap='magma')
        if body == 'moon':
            supported = subset.loc[(subset['region'] == region) & (subset['population'] == 'image_supported')]
            ax.scatter((supported['center_x_m'] - origin_x) / 1000,
                       (supported['center_y_m'] - origin_y) / 1000,
                       s=17, facecolors='none', edgecolors='#66c2a5', linewidths=.4)
        ax.set(title=region.replace('A3CLR22_', '').replace('_', ' '), xlabel='East from grid origin (km)',
               ylabel='North from grid origin (km)', aspect='equal')
        ax.set_xlim(0, grid['shape'][1] * .03)
        ax.set_ylim(-grid['shape'][0] * .03, 0)
    for ax in list(axes.flat)[len(regions):]:
        ax.set_visible(False)
    fig.suptitle(f'{body.title()}: eligible 600 m centers, 960 m stride; P90 slope; color capped at 40 degrees')
    fig.tight_layout(rect=(0, 0, .92, .95))
    fig.colorbar(ScalarMappable(norm=Normalize(0, 40), cmap='magma'), ax=list(axes.flat),
                 fraction=.018, pad=.02, label='P90 slope (degrees)')
    save_figure(fig, f'{body}_600m_center_maps')

# Findings and next decision

The research shortlist is provisional until development-site raster checks.
Final reserves are selected from independent context and kept out of patch EDA.
New Earth candidate locations alone cannot give an untouched planetary-query
test: reserve new planetary regions too, with overlap/source checks before use.

In [ ]:
# Read the Markdown notes below for the standalone report export.
report_notebook = json.loads((ROOT / 'notebooks/exploration/18_region_diversity_eda.ipynb').read_text(encoding="utf-8"))
report_notes = {entry["metadata"]["report_section"]: "".join(entry["source"])
                for entry in report_notebook["cells"]
                if entry["cell_type"] == "markdown" and "report_section" in entry["metadata"]}
report = [
    *report_notes['narrative-688c92ff-1'].split('\n'),
]
for (region, population), group in subset.groupby(['region', 'population']):
    report.append(f'| {region} / {population} | {len(group)} | {group["slope_p90_deg"].median():.2f} | '
                  f'{group["central_relief_m"].median():.2f} | {group["plane_residual_fraction"].median():.3f} |')
report += [
    *report_notes['narrative-688c92ff-2'].split('\n'),
]
for (population, scenario), group in exact_summary.loc[(exact_summary['size_m'] == 600)
                  & (exact_summary['region'] == 'ALL_QUERY_WEIGHTED')].groupby(['population', 'scenario']):
    numbers = group.set_index('earth_sites').loc[list(range(4)), 'pct'].to_numpy()
    report.append(f'| {population} | {scenario} | ' + ' | '.join(f'{v:.1f}' for v in numbers) + ' |')
report += [
    *report_notes['narrative-688c92ff-3'].split('\n'),
]
selection = bin_coverage.loc[(bin_coverage['size_m'] == 600) & (bin_coverage['population'] == 'image_supported')
                            & (bin_coverage['region'] == 'ALL_QUERY_WEIGHTED') & (bin_coverage['earth_sites'] >= 2)]
for (cohort, minimum), group in selection.groupby(['cohort', 'min_centers_per_site_bin']):
    values = group.groupby('scheme')['pct'].sum()
    report.append(f'| {cohort} | {minimum} | {values.min():.1f}–{values.max():.1f} |')
report += [
    *report_notes['narrative-688c92ff-4'].split('\n'),
    'Prioritize repeated examples of steeper, higher-relief, low-residual-fraction terrain in geographically separated Earth regions, with flat and rough controls retained. Verify terrain and source support in new development regions before increasing the collection. Do not choose final reserves by their candidate score or inspect their patch distributions to tune the model. See the [prospective collection plan](../../../EDA_PLAN.md) and [primary-source region research](../region-diversity-20261029/REGION_RESEARCH.md).',
    '',
    '## Artifacts',
    '',
    '- [Site inventory](site_inventory.csv), [descriptor quantiles](descriptor_quantiles.csv), [whole-site groups](spatial_groups.json).',
    '- [Terrain cells](terrain_bin_inventory.csv), [lunar cell replication](lunar_bin_replication.csv).',
    '- [Per-query site replication](query_site_replication.csv.gz), [exact-band summaries](exact_band_replication.csv).',
    '- [Physical distributions](600m_descriptor_distributions.png), [site replication](600m_exact_site_replication.png).',
    '- [Earth maps](earth_600m_center_maps.png), [Moon maps](moon_600m_center_maps.png).',
    '- [Provenance](provenance.json): input/output hashes, source hash, configuration and software.',
    *report_notes['narrative-688c92ff-5'].split('\n'),
]
(OUT / 'REPORT.md').write_text('\n'.join(report) + '\n', encoding='utf-8')
provenance['inputs'] = evidence
provenance['outputs'] = {str(p.relative_to(OUT)): file_hash(p) for p in sorted(OUT.rglob('*'))
                         if p.is_file() and p.name != 'provenance.json'}
provenance['status'] = 'complete'
(OUT / 'provenance.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')
display(Markdown('\n'.join(report)))
print('Complete:', OUT)

# Terrain diversity inventory

This inventory informs multi-region acquisition and evaluation design. No model was trained, and neither coarse bins nor descriptor bands are validated analog labels.

## Represented collection

Three Earth sites and seven lunar south-pole regions; all are exploratory development data. Earth source family is FABDEM V1-2 with WorldCover 2021 screening; Moon is PGDA104 SfS/LOLA. Body and source family are confounded. Geographic separation does not establish statistical independence, and source differences cannot be removed by counting sites. Mars is absent.

There are 28,284 eligible center/size/view records and 23,205 common-960 records. The same 960 m center lattice is used. Eligible-at-size relaxes the complete-960 window restriction; it does not enumerate every eligible patch. Overlapping scales and support views stay with their site.

## 600 m physical distributions

| Site / population | Eligible centers | Median P90 slope (deg) | Median central relief (m) | Median plane residual fraction |
| --- | ---: | ---: | ---: | ---: |


The strict lunar populations emphasize steeper, higher-relief patches close to a plane relative to the typical retained Earth patch. Plane-residual fraction is normalized by elevation SD; low fractions do not establish low absolute roughness, safe traversal or physical smoothness independent of reconstruction. Absolute RMS is exported alongside it.

## Existing bands: how many Earth sites supply a counterpart?

Common-960 cohort, 600 m, query-weighted within each support population:

| Population | Scenario | 0 sites (%) | 1 site (%) | 2 sites (%) | 3 sites (%) |
| --- | --- | ---: | ---: | ---: | ---: |


These rows reuse notebook 17’s fixed physical bands and verified site-specific counts. They measure collection replication under those bands, not generalization or retrieval accuracy.

## Coarse descriptor-cell sensitivity

Cross P90 slope / central relief / plane fraction into 48 cells. Coarse breaks are 5,15,30 degrees / 25,75,200 m / .25,.5; shifted breaks are 7.5,20,35 degrees / 40,100,250 m / .2,.4. Internal boundaries enter the upper cell. Cells are exploratory descriptors, not landform or analog classes. Broad cells can include poor matches; they complement the stricter band analysis.

| Cohort | Minimum centers per Earth site/cell | Strict lunar centers in cells represented at ≥2 Earth sites (%) across two bin schemes |
| --- | ---: | ---: |


The 1/5/20-center scenarios expose sparse-bin sensitivity; no count is a sufficiency threshold or independent sample size. Cohort comparisons change both Earth and lunar populations, so their percentage differences cannot isolate the effect of admitting Earth candidates alone. Per-region and all-size tables are preserved; pooled counts can conceal regional variation.

## Decision



Native source accuracy, access, environmental suitability, relevance labels and model performance are not established by this inventory. Grid spacing remains distinct from effective resolution. Proposed regions are hypotheses to measure, not confirmed gap-fillers.